# Modem Telemetry Anomaly Detection — Pipeline

**Structure**
- **0. Configuration** — paths, constants, parameters (change things here only)
- **1. Load & combine** the three observation windows
- **2. Data quality** — raw null handling
- **3. Exploratory distributions**
- **4. Feature engineering** — expanding per-modem baselines, deviations, warm-up
- **5. Deviation null handling**
- **6. Model** — temporal train/test split, scaling, Isolation Forest, contamination cut-offs
- **7. Reference labels** — operational thresholds (evaluated on test)
- **8. Detection rules** — 3-consecutive (primary), 3-of-5 (sensitivity)
- **9. Lead time — Isolation Forest** (units, 5 outcome categories, 10 h early bound, persistence)
- **10. Algorithm comparison** — elbow for *k*, k-means and LOF under identical conditions
- **11. Hypothesis tests** — chi-square + Cramér's V, McNemar, Wilcoxon (Bonferroni)
- **12. Figures**
- **13. Sensitivity analysis** — contamination levels × detection rules
- **14. Missed degradation**
- **15. Retention log & save results**
- **16. Exploratory (post hoc)** — sustained breaches, alert efficiency
- **Appendix** — diagnostics that justify the design decisions

## 0. Configuration

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import os
from sklearn.ensemble import IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import LocalOutlierFactor
from scipy.stats import chi2_contingency, wilcoxon
from statsmodels.stats.contingency_tables import mcnemar   # pip install statsmodels

BASE = "./data/"                       # raw CSV files exported from the modem database
RESULTS_DIR = BASE + "results/"          # all tables (CSV) and figures (PNG) for the thesis
os.makedirs(RESULTS_DIR, exist_ok=True)

# file -> (window label, has header row)
FILES = {
    "modem_features.csv":     ("W1_aug",   True),
    "modem_features0903.csv": ("W2_sep01", False),   # exported with --silent -> no header
    "modem_features0910.csv": ("W3_sep08", False),
}
RAW_COLS = ['modem_key', 'datetime', 'timestamp_unix', 'latencyavg',
            'utxmin', 'utxmax', 'uchn', 'usnrmin', 'drxmin', 'drxmax', 'dchn']

# Baselines, warm-up and detection rules never cross an observation-window gap
GROUP_KEYS = ['modem_key', 'window']
WARMUP_INTERVALS = 12            # 1 hour of 5-min readings per modem per window

# Signal features (channel counts uchn/dchn excluded: hardware config, not signal quality)
FEATURE_COLS = ['latencyavg', 'utxmin', 'utxmax', 'usnrmin', 'drxmin', 'drxmax']
STD_DEV_COLS = ['latencyavg', 'usnrmin', 'drxmin', 'drxmax']   # genuine per-modem variance -> z-score
ABS_DEV_COLS = ['utxmin', 'utxmax']                            # step-function behaviour -> signed difference
DEV_COLS     = [f'{c}_dev' for c in FEATURE_COLS]

# Absolute latency excluded from the model: not comparable across modem models (CM3500B/EPC3949)
ABS_MODEL_FEATURES = ['utxmin', 'utxmax', 'usnrmin', 'drxmin', 'drxmax']
MODEL_FEATURES     = ABS_MODEL_FEATURES + DEV_COLS

# Isolation Forest (parameters per Liu et al. 2008)
N_ESTIMATORS = 100
MAX_SAMPLES  = 256
RANDOM_STATE = 42
CONTAMINATION_LEVELS = [0.01, 0.03, 0.05]
PRIMARY_FLAG = 'flag_iso_03'
PRIMARY_CONTAMINATION = 0.03
# Training rows used to set every algorithm's flag threshold (the SAME rows for all algorithms)
THRESHOLD_SAMPLE = 1_000_000

# Comparison algorithms (Section 9.5)
K_RANGE        = range(2, 16)   # candidate k for the elbow method
ELBOW_SAMPLE   = 200_000        # training rows used for the elbow curve
K_CLUSTERS     = None           # None -> chosen automatically from the elbow curve; set an int to override
LOF_NEIGHBORS  = 20
LOF_FIT_SAMPLE = 50_000         # LOF is fitted on a random training sample (computational cost)

# Evaluation (Sections 9.4 and 9.6)
EARLY_BOUND_MINUTES = 600       # 10 h, agreed with the support manager
OUTCOMES = ['early_within', 'early_beyond', 'concurrent', 'late', 'not_detected']
OUTCOME_LABELS = {'early_within': 'Early (within 10 h)', 'early_beyond': 'Early (beyond 10 h)',
                  'concurrent': 'Concurrent', 'late': 'Late', 'not_detected': 'Not detected'}
ALPHA = 0.05
N_PAIRWISE_TESTS = 4            # 2 x McNemar + 2 x Wilcoxon
ALPHA_PAIRWISE = ALPHA / N_PAIRWISE_TESTS   # Bonferroni: 0.0125

# Temporal hold-out: train on the earlier windows, test on the latest (mirrors deployment)
TRAIN_WINDOWS = ['W1_aug', 'W2_sep01']
TEST_WINDOWS  = ['W3_sep08']

# Stig's operational thresholds (values stored in tenths: 490 = 49.0)
STIG_UTX_MAX       = 490
STIG_USNR_MIN      = 250
STIG_DRX_ABS       = 100
STIG_LATENCY_FACTOR = 2
STIG_LATENCY_MODE  = 'global'    # 'global' = 2 x population median (current); 'per_modem' = 2 x the modem's own baseline

# Figure colours (validated categorical palette; labels carry identity, never colour alone)
COL_OUTCOME = {'early_within': '#2a78d6', 'early_beyond': '#eb6834', 'concurrent': '#1baf7a',
               'late': '#eda100', 'not_detected': '#c3c2b7'}
COL_ALGO = {'IsolationForest': '#2a78d6', 'KMeans': '#eb6834', 'LOF': '#1baf7a'}
INK, INK_MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'axes.edgecolor': INK_MUTED, 'axes.labelcolor': INK, 'xtick.color': INK_MUTED,
                     'ytick.color': INK_MUTED, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'font.size': 10, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

def save_table(df, name):
    df.to_csv(RESULTS_DIR + name + ".csv")
    return df

# Retention log: row/modem counts after every filtering step (table for the thesis)
retention_log = []
def log_step(step, frame, note=""):
    retention_log.append({'step': step, 'rows': len(frame),
                          'modems': frame['modem_key'].nunique(), 'note': note})
    print(f"[{step}] rows={len(frame):,}  modems={frame['modem_key'].nunique():,}  {note}")

## 1. Load & combine observation windows

In [ ]:
parts = []
for f, (window, has_header) in FILES.items():
    part = pd.read_csv(BASE + f, sep='\t', encoding='utf-16',
                       header=0 if has_header else None,
                       names=None if has_header else RAW_COLS)
    part['window'] = window
    parts.append(part)
    print(f"{f:<25} {window:<9} {len(part):>10,} rows  {part['datetime'].min()} -> {part['datetime'].max()}")

df = pd.concat(parts, ignore_index=True)
df['datetime'] = pd.to_datetime(df['datetime'])
del parts

n_before = len(df)
df = df.drop_duplicates(subset=['modem_key', 'timestamp_unix'])
log_step('1_loaded', df, f"{n_before - len(df):,} duplicates removed")

print(df.groupby('window').agg(start=('datetime', 'min'), end=('datetime', 'max'),
                               rows=('modem_key', 'size'), modems=('modem_key', 'nunique')))

## 2. Data quality — raw nulls
Rows with a missing signal value are dropped (sporadic measurement failures, <0.3 % of rows).
Justification analysis: see **Appendix A1**.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[lambda s: s > 0])

df_clean = df.dropna().copy()
log_step('2_raw_nulls_dropped', df_clean,
         f"{len(df) - len(df_clean):,} rows dropped, retention {len(df_clean)/len(df)*100:.2f}%")

## 3. Exploratory distributions (thesis figure)

In [ ]:
plot_cols = ['latencyavg', 'utxmin', 'utxmax', 'uchn', 'usnrmin', 'drxmin', 'drxmax', 'dchn']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), plot_cols):
    ax.hist(df_clean[col], bins=50, color='black', alpha=0.7)
    ax.set_title(col); ax.set_xlabel('Value'); ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(BASE + "feature_distributions.png", dpi=150)
plt.show()

del df   # raw frame no longer needed

## 4. Feature engineering
- Expanding per-modem baseline (mean/std up to and including the current row), **per window**
- Standardised deviation for variable features, signed absolute deviation for UsTX (Appendix A2)
- First `WARMUP_INTERVALS` readings per modem per window excluded (baseline not yet stable)

In [ ]:
start = time.time()
df_feat = df_clean.sort_values(GROUP_KEYS + ['datetime']).copy()
del df_clean

grouped = df_feat.groupby(GROUP_KEYS)
for col in FEATURE_COLS:
    df_feat[f'{col}_exp_mean'] = grouped[col].transform(lambda s: s.expanding().mean())
    df_feat[f'{col}_exp_std']  = grouped[col].transform(lambda s: s.expanding().std())

for col in STD_DEV_COLS:
    std = df_feat[f'{col}_exp_std'].replace(0, np.nan)          # undefined if no variance yet
    df_feat[f'{col}_dev'] = (df_feat[col] - df_feat[f'{col}_exp_mean']) / std

for col in ABS_DEV_COLS:
    df_feat[f'{col}_dev'] = df_feat[col] - df_feat[f'{col}_exp_mean']   # signed: direction matters

df_feat['observation_index'] = df_feat.groupby(GROUP_KEYS).cumcount()
df_model = df_feat[df_feat['observation_index'] >= WARMUP_INTERVALS].copy()
log_step('4_warmup_excluded', df_model, f"first {WARMUP_INTERVALS} intervals per modem-window")
print(f"Feature engineering took {time.time() - start:.0f}s")

## 5. Deviation null handling
Deviation is undefined where a metric has not yet varied within a modem-window. Those rows are dropped.
Which modems disappear entirely is recorded for the write-up (details: Appendix A3).

In [ ]:
print("Deviation nulls per feature:")
print(df_model[DEV_COLS].isnull().sum())

modems_before = set(df_model['modem_key'].unique())
pre_drop = df_model                                   # reference kept briefly for the loss breakdown
df_model = df_model.dropna(subset=DEV_COLS).copy()
lost_modems = modems_before - set(df_model['modem_key'].unique())

log_step('5_deviation_nulls_dropped', df_model, f"{len(lost_modems)} modems lost entirely")
print("\nNull deviation rows inside the lost modems:")
print(pre_drop[pre_drop['modem_key'].isin(lost_modems)][DEV_COLS].isnull().sum())

del pre_drop, df_feat

## 6. Model — temporal split, scaling, Isolation Forest
- **Train:** W1 + W2 (Aug 5–7, Sep 1–2) · **Test:** W3 (Sep 8–9)
- Scaler, model **and** contamination thresholds are fitted on train only and applied unchanged to test
- Random row splits are unsuitable: they break the per-modem time series needed for the detection rules and lead time

In [ ]:
df_model['split'] = np.where(df_model['window'].isin(TEST_WINDOWS), 'test', 'train')
train = df_model['split'] == 'train'
log_step('6_train_split', df_model[train], f"windows {TRAIN_WINDOWS}")
log_step('6_test_split',  df_model[~train], f"windows {TEST_WINDOWS}")
print(f"Train share: {train.mean()*100:.1f}%  |  Test share: {(~train).mean()*100:.1f}%")

# Fixed sample of training rows used to set every algorithm's flag threshold (same rows for all)
rng = np.random.default_rng(RANDOM_STATE)
THR_INDEX = rng.choice(df_model.index[train].to_numpy(), size=min(THRESHOLD_SAMPLE, int(train.sum())), replace=False)

# Fit scaler + model on TRAIN only
scaler = StandardScaler().fit(df_model.loc[train, MODEL_FEATURES])

start = time.time()
iso = IsolationForest(n_estimators=N_ESTIMATORS, max_samples=MAX_SAMPLES,
                      contamination='auto', random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(scaler.transform(df_model.loc[train, MODEL_FEATURES]))

# Score all rows (train scores needed for thresholds, test scores for evaluation)
df_model['anomaly_score'] = iso.score_samples(scaler.transform(df_model[MODEL_FEATURES]))   # lower = more anomalous
IF_RUNTIME = time.time() - start
print(f"Training + scoring took {IF_RUNTIME:.1f}s")

In [ ]:
# Thresholds from TRAIN score distribution, applied unchanged to TEST.
# If the test flag share lands near 1/3/5 %, the model's notion of "anomalous" transfers to unseen data.
rows = []
for c in CONTAMINATION_LEVELS:
    thr = df_model.loc[THR_INDEX, 'anomaly_score'].quantile(c)
    col = f'flag_iso_{int(c*100):02d}'
    df_model[col] = df_model['anomaly_score'] <= thr
    rows.append({'contamination': c, 'threshold': round(thr, 4),
                 'train_flagged_pct': df_model.loc[train, col].mean() * 100,
                 'test_flagged_pct':  df_model.loc[~train, col].mean() * 100,
                 'test_flagged_modems': df_model.loc[~train & df_model[col], 'modem_key'].nunique()})
contamination_table = save_table(pd.DataFrame(rows).round(2), 'T1_contamination_generalisation')
contamination_table

## 7. Reference labels — operational thresholds
Kept **outside** the model: used only for validation and as the deterioration point for lead time.
Flags are computed on all rows; **reported figures use the test window only**.

`STIG_LATENCY_MODE` switches the latency rule between a population-wide median (`'global'`) and the modem's own
baseline (`'per_modem'`, closer to "should not increase by 100 %"). Latency breaches are rare either way.

In [ ]:
def add_stig_flags(df):
    if STIG_LATENCY_MODE == 'per_modem':
        # the modem's own baseline BEFORE the current interval (no look-ahead)
        latency_ref = df.groupby(GROUP_KEYS, sort=False)['latencyavg_exp_mean'].shift(1)
    else:
        latency_ref = df['latencyavg'].median()
    df['stig_utx']     = (df['utxmin'] > STIG_UTX_MAX) | (df['utxmax'] > STIG_UTX_MAX)
    df['stig_usnr']    = df['usnrmin'] < STIG_USNR_MIN
    df['stig_drx']     = (df['drxmin'] < -STIG_DRX_ABS) | (df['drxmax'] > STIG_DRX_ABS)
    df['stig_latency'] = df['latencyavg'] > latency_ref * STIG_LATENCY_FACTOR
    df['stig_any']     = df[['stig_utx', 'stig_usnr', 'stig_drx', 'stig_latency']].any(axis=1)
    return df

df_model = df_model.sort_values(GROUP_KEYS + ['datetime'])
df_model = add_stig_flags(df_model)
df_eval = df_model[df_model['split'] == 'test']

print("Test rows breaching each threshold:")
print(df_eval[['stig_utx', 'stig_usnr', 'stig_drx', 'stig_latency', 'stig_any']].sum())

stig_total = df_eval['stig_any'].sum()
overlap_table = pd.DataFrame([{
    'flag': c,
    'overlap_rows': (df_eval[c] & df_eval['stig_any']).sum(),
    'pct_model_flags_breaching_stig': (df_eval[c] & df_eval['stig_any']).sum() / df_eval[c].sum() * 100,
    'pct_stig_breaches_caught':       (df_eval[c] & df_eval['stig_any']).sum() / stig_total * 100,
} for c in ['flag_iso_01', 'flag_iso_03', 'flag_iso_05']]).round(1)
save_table(overlap_table, 'T2_agreement_with_reference_IF')

## 8. Detection rules
- **Primary:** 3 consecutive flagged intervals (15 min, per Stig)
- **Sensitivity:** 3 of the last 5 intervals (captures intermittent degradation)

Rolling windows are grouped per modem-window so no rule spans a data gap. Table shows the **test window**.

In [ ]:
# Rolling windows are computed within each modem-window, so a rule can never span the gap between windows.
def rule_consecutive(df, flag_col, n=3):
    """Detection fires when flag_col is True for n consecutive intervals."""
    s = df.groupby(GROUP_KEYS)[flag_col].transform(lambda x: x.astype(int).rolling(n, min_periods=n).sum())
    return s == n

def rule_n_of_m(df, flag_col, n=3, m=5):
    """Detection fires when flag_col is True for at least n of the last m intervals."""
    s = df.groupby(GROUP_KEYS)[flag_col].transform(lambda x: x.astype(int).rolling(m, min_periods=m).sum())
    return s >= n

df_model = df_model.sort_values(GROUP_KEYS + ['datetime'])

for c in ['flag_iso_01', 'flag_iso_03', 'flag_iso_05']:
    df_model[f'{c}_3consec'] = rule_consecutive(df_model, c)
    df_model[f'{c}_3of5']    = rule_n_of_m(df_model, c)

df_eval = df_model[df_model['split'] == 'test']   # refresh: now includes the rule columns

rows = []
for c in ['flag_iso_01', 'flag_iso_03', 'flag_iso_05']:
    for variant in [c, f'{c}_3consec', f'{c}_3of5']:
        rows.append({'flag': c, 'rule': variant.replace(c, '') or '_raw',
                     'rows': df_eval[variant].sum(),
                     'modems': df_eval.loc[df_eval[variant], 'modem_key'].nunique()})
rule_table = pd.DataFrame(rows).pivot(index='flag', columns='rule', values=['rows', 'modems'])
save_table(rule_table, 'T3_detection_rule_effect')

## 9. Lead time — Isolation Forest
**Units:** every test modem-window that breaches an operational threshold at least once (identical for all algorithms).
Lead time *L* = first breach − first detection (minutes; positive = model earlier).

| Outcome | Definition |
|---|---|
| Early within bound | 0 < *L* ≤ 600 min |
| Early beyond bound | *L* > 600 min (false positive, Saxena et al.) |
| Concurrent | *L* = 0 |
| Late | *L* < 0 |
| Not detected | no detection in the window |

**Persistence** = share of intervals from the first detection up to the breach in which the algorithm kept flagging.

In [ ]:
def first_time(df, mask):
    """First timestamp per modem-window where mask is True."""
    return df.loc[mask, GROUP_KEYS + ['datetime']].groupby(GROUP_KEYS)['datetime'].min()

# Units: all test modem-windows with at least one threshold breach
units = first_time(df_eval, df_eval['stig_any']).rename('breach_time').to_frame()
print(f"Units (test modem-windows with >= 1 threshold breach): {len(units):,} "
      f"of {df_eval.groupby(GROUP_KEYS).ngroups:,} test modem-windows")

def classify(lead_min):
    out = pd.Series('not_detected', index=lead_min.index, dtype=object)
    out[lead_min < 0] = 'late'
    out[lead_min == 0] = 'concurrent'
    out[(lead_min > 0) & (lead_min <= EARLY_BOUND_MINUTES)] = 'early_within'
    out[lead_min > EARLY_BOUND_MINUTES] = 'early_beyond'
    return pd.Categorical(out, categories=OUTCOMES)

def persistence(test, flag, det_time, breach_time):
    """Share of intervals in [first detection, breach) that are still flagged (early detections only)."""
    early = pd.DataFrame({'det': det_time, 'breach': breach_time}).dropna()
    early = early[early['breach'] > early['det']].reset_index()
    if early.empty:
        return pd.Series(dtype=float)
    m = test[GROUP_KEYS + ['datetime']].assign(_f=flag.values).merge(early, on=GROUP_KEYS)
    m = m[(m['datetime'] >= m['det']) & (m['datetime'] < m['breach'])]
    return m.groupby(GROUP_KEYS)['_f'].mean()

def evaluate_detector(test, flag, detection, name, unit_frame=None):
    """Outcome per unit for one detector. flag = per-interval flag, detection = after the detection rule."""
    u = (units if unit_frame is None else unit_frame).copy()
    u['det_time'] = first_time(test, detection).reindex(u.index)
    u['lead_min'] = (u['breach_time'] - u['det_time']).dt.total_seconds() / 60
    u['outcome'] = classify(u['lead_min'])
    u['persistence'] = persistence(test, flag, u['det_time'], u['breach_time']).reindex(u.index)
    u['detector'] = name
    return u

def summarise(u, name=None):
    det = u['outcome'] != 'not_detected'
    m = u.loc[det, 'lead_min']
    shares = u['outcome'].value_counts(normalize=True).reindex(OUTCOMES).mul(100)
    return {'detector': name or u['detector'].iloc[0], 'units': len(u), 'detected_pct': det.mean() * 100,
            'median_lead_min': m.median(), 'q25_lead_min': m.quantile(.25), 'q75_lead_min': m.quantile(.75),
            **{f'{o}_pct': shares[o] for o in OUTCOMES},
            'median_persistence_early_within': u.loc[u['outcome'] == 'early_within', 'persistence'].median()}

# Isolation Forest, primary flag rate: raw flag vs. the two detection rules
f = PRIMARY_FLAG
u_if_raw  = evaluate_detector(df_eval, df_eval[f], df_eval[f],               'IF raw flag')
u_if      = evaluate_detector(df_eval, df_eval[f], df_eval[f'{f}_3consec'],  'IF 3-consecutive')
u_if_3of5 = evaluate_detector(df_eval, df_eval[f], df_eval[f'{f}_3of5'],     'IF 3-of-5')

lead_table = save_table(pd.DataFrame([summarise(u) for u in (u_if_raw, u_if, u_if_3of5)])
                        .set_index('detector').round(1), 'T4_lead_time_IF')
lead_table

### 9.1 How much earlier? — early detections by lead time
Answers the concrete question (30 min, 3 h, 3 days?). Credible early warnings combine a lead within the bound
with **high persistence**; long leads with persistence near zero are likely unrelated events.

In [ ]:
def bucket_table_for(u):
    early = u[u['lead_min'] > 0].copy()
    bins   = [0, 30, 60, 180, 360, EARLY_BOUND_MINUTES, np.inf]
    labels = ['<= 30 min', '30-60 min', '1-3 h', '3-6 h', '6-10 h', '> 10 h\n(beyond bound)']
    early['bucket'] = pd.cut(early['lead_min'], bins=bins, labels=labels)
    t = early.groupby('bucket', observed=False).agg(modem_windows=('lead_min', 'size'),
                                                    median_lead_min=('lead_min', 'median'),
                                                    median_persistence=('persistence', 'median'))
    t['share_of_early_pct'] = t['modem_windows'] / max(len(early), 1) * 100
    t['share_of_all_units_pct'] = t['modem_windows'] / len(u) * 100
    return t.round(2)

bucket_table = save_table(bucket_table_for(u_if), 'T5_early_detection_buckets_IF')
bucket_table

## 10. Algorithm comparison — Isolation Forest vs. k-means vs. LOF
Everything except the algorithm is held constant: same 11 features, same scaler (fitted on train), same
training windows, same threshold rows (`THR_INDEX`), same flag rate, same detection rule, same reference, same units.

### 10.1 Choosing *k* for k-means (elbow method)

In [ ]:
train = df_model['split'] == 'train'
rng = np.random.default_rng(RANDOM_STATE)
elbow_idx = rng.choice(df_model.index[train].to_numpy(), size=min(ELBOW_SAMPLE, int(train.sum())), replace=False)
X_elbow = scaler.transform(df_model.loc[elbow_idx, MODEL_FEATURES])

inertia = []
for k in K_RANGE:
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, batch_size=10_000, n_init=3).fit(X_elbow)
    inertia.append(km.inertia_)
inertia = np.array(inertia)

# Knee = point with the largest distance to the straight line between the first and last point (normalised)
ks = np.array(list(K_RANGE), dtype=float)
xn = (ks - ks.min()) / (ks.max() - ks.min())
yn = (inertia - inertia.min()) / (inertia.max() - inertia.min())
k_auto = int(ks[np.argmax(np.abs(yn - (1 - xn)))])
K = K_CLUSTERS or k_auto
print(f"Elbow suggests k = {k_auto}; using k = {K}")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(ks, inertia, color=COL_ALGO['KMeans'], linewidth=2, marker='o', markersize=5)
ax.axvline(K, color=INK_MUTED, linestyle='--', linewidth=1)
ax.annotate(f'k = {K}', xy=(K, inertia[list(K_RANGE).index(K)]), xytext=(8, 10),
            textcoords='offset points', color=INK)
ax.set_xlabel('Number of clusters k'); ax.set_ylabel('Within-cluster sum of squares')
ax.set_title('Elbow curve for k-means (training sample)', loc='left', color=INK)
plt.savefig(RESULTS_DIR + "F1_elbow_kmeans.png"); plt.show()
del X_elbow

### 10.2 Train, score and evaluate all three algorithms

In [ ]:
X_thr  = scaler.transform(df_model.loc[THR_INDEX, MODEL_FEATURES])   # threshold rows (same for all)
X_test = scaler.transform(df_eval[MODEL_FEATURES])
base = df_eval[GROUP_KEYS + ['datetime', 'stig_any']].copy()          # sorted by modem, window, time

def score_in_chunks(score_fn, X, chunk=500_000, label=''):
    out = np.empty(len(X))
    for i in range(0, len(X), chunk):
        out[i:i + chunk] = score_fn(X[i:i + chunk])
        print(f"  {label}: scored {min(i + chunk, len(X)):,} / {len(X):,}", end='\r')
    print()
    return out

def run_detector(name, thr_scores, test_scores, runtime):
    """Flag with the train-sample quantile, apply the primary rule, evaluate on the common units."""
    thr = np.quantile(thr_scores, PRIMARY_CONTAMINATION)
    d = base.copy()
    d['flag'] = test_scores <= thr
    d['det'] = rule_consecutive(d, 'flag')
    test_flags[name] = d[['flag', 'det']]          # kept for the exploratory analysis (Section 16)
    u = evaluate_detector(d, d['flag'], d['det'], name)
    both = (d['flag'] & d['stig_any']).sum()
    row = {**summarise(u, name),
           'test_flag_pct': d['flag'].mean() * 100,
           'pct_flags_breaching': both / d['flag'].sum() * 100,
           'pct_breaches_flagged': both / d['stig_any'].sum() * 100,
           'modems_detected': d.loc[d['det'], 'modem_key'].nunique(),
           'runtime_s': runtime}
    return u, row

results, rows, test_flags = {}, [], {}

# Isolation Forest (already trained in Section 6)
results['IsolationForest'], r = run_detector('IsolationForest', iso.score_samples(X_thr),
                                             df_eval['anomaly_score'].to_numpy(), IF_RUNTIME)
rows.append(r)

# k-means: anomaly score = negative distance to the nearest centre (lower = more anomalous)
t = time.time()
X_train = scaler.transform(df_model.loc[train, MODEL_FEATURES])
km = MiniBatchKMeans(n_clusters=K, random_state=RANDOM_STATE, batch_size=10_000, n_init=3).fit(X_train)
del X_train
km_score = lambda X: -km.transform(X).min(axis=1)
km_test = score_in_chunks(km_score, X_test, label='k-means')
results['KMeans'], r = run_detector('KMeans', km_score(X_thr), km_test, time.time() - t)
rows.append(r)

# Local Outlier Factor, fitted on a random training sample (novelty mode -> scores unseen data)
t = time.time()
lof_idx = rng.choice(df_model.index[train].to_numpy(), size=min(LOF_FIT_SAMPLE, int(train.sum())), replace=False)
lof = LocalOutlierFactor(n_neighbors=LOF_NEIGHBORS, novelty=True, n_jobs=-1)
lof.fit(scaler.transform(df_model.loc[lof_idx, MODEL_FEATURES]))
lof_test = score_in_chunks(lof.score_samples, X_test, label='LOF')
results['LOF'], r = run_detector('LOF', score_in_chunks(lof.score_samples, X_thr, label='LOF thr'),
                                 lof_test, time.time() - t)
rows.append(r)

# Save test scores so later analyses can run without re-scoring LOF
pd.DataFrame({'IsolationForest': df_eval['anomaly_score'].to_numpy(), 'KMeans': km_test, 'LOF': lof_test},
             index=df_eval.index).to_parquet(BASE + "test_scores_by_algorithm.parquet")

comparison = save_table(pd.DataFrame(rows).set_index('detector').round(2), 'T6_algorithm_comparison')
comparison

## 11. Hypothesis tests
- **H1 (overall):** chi-square test of independence on the 3 × 5 outcome table, effect size Cramér's V. α = 0.05.
  The same units appear under all algorithms, so observations are not independent (Section 9.7) — interpreted with care.
- **H1 (pairwise):** McNemar test, IF vs. each comparison algorithm, outcome *early within bound* yes/no. α = 0.0125.
- **H2:** Wilcoxon signed-rank test on paired lead times of units detected by both algorithms. α = 0.0125.
  Effect sizes: median paired difference and matched-pairs rank-biserial correlation (positive = IF earlier).

In [ ]:
# ---- H1 overall: chi-square
counts = pd.DataFrame({n: u['outcome'].value_counts().reindex(OUTCOMES, fill_value=0)
                       for n, u in results.items()}).T
counts_used = counts.loc[:, counts.sum() > 0]
chi2, p_chi2, dof, _ = chi2_contingency(counts_used)
cramers_v = np.sqrt(chi2 / (counts_used.values.sum() * (min(counts_used.shape) - 1)))
save_table(counts, 'T7_outcome_counts')
print(f"Chi-square: chi2 = {chi2:.2f}, dof = {dof}, p = {p_chi2:.3g}, Cramer's V = {cramers_v:.3f} "
      f"-> H0,1 {'rejected' if p_chi2 < ALPHA else 'not rejected'} at alpha = {ALPHA}")

# ---- H1 pairwise: McNemar on "early within bound"
def mcnemar_row(a_name, b_name):
    a = results[a_name]['outcome'] == 'early_within'
    b = (results[b_name]['outcome'] == 'early_within').reindex(a.index)
    tab = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
    exact = (tab[0][1] + tab[1][0]) < 25
    res = mcnemar(tab, exact=exact, correction=not exact)
    return {'comparison': f'{a_name} vs {b_name}', 'test': 'McNemar (early within bound)',
            'both': tab[0][0], f'only_{a_name}': tab[0][1], 'only_other': tab[1][0], 'neither': tab[1][1],
            'statistic': res.statistic, 'p_value': res.pvalue,
            'effect': f'odds ratio of discordant pairs = {tab[0][1] / max(tab[1][0], 1):.2f}',
            'reject_H0': res.pvalue < ALPHA_PAIRWISE}

# ---- H2: Wilcoxon on paired lead times
def wilcoxon_row(a_name, b_name):
    a, b = results[a_name], results[b_name].reindex(results[a_name].index)
    both = a['det_time'].notna() & b['det_time'].notna()
    d = (a.loc[both, 'lead_min'] - b.loc[both, 'lead_min'])
    nz = d[d != 0]
    if len(nz) == 0:
        stat, p, rrb = np.nan, 1.0, 0.0
    else:
        stat, p = wilcoxon(a.loc[both, 'lead_min'], b.loc[both, 'lead_min'], zero_method='wilcox')
        ranks = nz.abs().rank()
        rrb = (ranks[nz > 0].sum() - ranks[nz < 0].sum()) / ranks.sum()
    return {'comparison': f'{a_name} vs {b_name}', 'test': 'Wilcoxon signed-rank (lead time)',
            'paired_units': int(both.sum()), 'statistic': stat, 'p_value': p,
            'median_diff_min': d.median(), 'rank_biserial': rrb,
            'effect': f'median diff = {d.median():.1f} min, r_rb = {rrb:.3f}',
            'reject_H0': p < ALPHA_PAIRWISE}

tests = pd.DataFrame([mcnemar_row('IsolationForest', 'KMeans'), mcnemar_row('IsolationForest', 'LOF'),
                      wilcoxon_row('IsolationForest', 'KMeans'), wilcoxon_row('IsolationForest', 'LOF')])
print(f"Pairwise alpha (Bonferroni, {N_PAIRWISE_TESTS} tests) = {ALPHA_PAIRWISE}")
save_table(tests, 'T8_pairwise_tests')
tests[['comparison', 'test', 'statistic', 'p_value', 'effect', 'reject_H0']]

## 12. Figures (saved to `RESULTS_DIR`)
- **F2** — outcome shares per algorithm (stacked bar, direct labels)
- **F3** — cumulative lead-time distribution per algorithm (detected units, clipped at ±10 h)
- **F4** — Isolation Forest early detections by lead time, with median persistence

In [ ]:
# ---- F2: outcome shares per algorithm
share = counts.div(counts.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(len(share))
for o in OUTCOMES:
    vals = share[o].to_numpy()
    ax.barh(share.index, vals, left=left, color=COL_OUTCOME[o], edgecolor='white', linewidth=2,
            height=0.6, label=OUTCOME_LABELS[o])
    for y, (l, v) in enumerate(zip(left, vals)):
        if v >= 5:
            ax.text(l + v / 2, y, f'{v:.0f}%', ha='center', va='center', fontsize=9,
                    color='white' if o in ('early_within', 'early_beyond') else INK)
    left += vals
ax.set_xlim(0, 100); ax.set_xlabel('Share of units (%)'); ax.invert_yaxis()
ax.grid(axis='y', visible=False)
ax.set_title('Detection outcome by algorithm (test window, 3 % flag rate, 3 consecutive intervals)',
             loc='left', color=INK, fontsize=10)
ax.legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.22), frameon=False, fontsize=9)
plt.savefig(RESULTS_DIR + "F2_outcome_shares.png"); plt.show()

# ---- F3: cumulative lead-time distribution
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for name, u in results.items():
    v = (u['lead_min'].dropna().clip(-EARLY_BOUND_MINUTES, EARLY_BOUND_MINUTES) / 60).sort_values()
    yv = np.arange(1, len(v) + 1) / len(v)
    ax.plot(v.values, yv, color=COL_ALGO[name], linewidth=2, label=name)
ax.axvline(0, color=INK_MUTED, linestyle='--', linewidth=1)
ax.text(5, 0.04, 'model earlier →', color=INK_MUTED, fontsize=9, ha='center')
ax.text(-5, 0.04, '← model later', color=INK_MUTED, fontsize=9, ha='center')
ax.set_xlim(-10.5, 10.5); ax.set_ylim(0, 1.02)
ax.set_xlabel('Lead time (hours; clipped at ±10 h)'); ax.set_ylabel('Cumulative share of detected units')
ax.set_title('Lead time distribution by algorithm (test window)', loc='left', color=INK, fontsize=10)
ax.legend(frameon=False, loc='upper left', fontsize=9)
plt.savefig(RESULTS_DIR + "F3_lead_time_cdf.png"); plt.show()

# ---- F4: IF early detections by lead time bucket, persistence as label
bt = bucket_table
fig, ax = plt.subplots(figsize=(8, 3.8))
colors = [COL_OUTCOME['early_beyond'] if '>' in str(b) else COL_OUTCOME['early_within'] for b in bt.index]
bars = ax.bar(bt.index.astype(str), bt['modem_windows'], color=colors, width=0.6, edgecolor='white', linewidth=2)
for bar, n, pers in zip(bars, bt['modem_windows'], bt['median_persistence']):
    lbl = f'{int(n)}' + ('' if pd.isna(pers) else f'\npersist. {pers:.2f}')
    ax.annotate(lbl, xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 3),
                textcoords='offset points', ha='center', va='bottom', fontsize=8.5, color=INK)
ax.set_ylabel('Modem-windows'); ax.set_xlabel('Lead time before threshold breach')
ax.grid(axis='x', visible=False)
ax.set_ylim(0, bt['modem_windows'].max() * 1.3 + 1)
ax.set_title('Isolation Forest: early detections by lead time (label: median persistence)',
             loc='left', color=INK, fontsize=10)
plt.savefig(RESULTS_DIR + "F4_early_detection_buckets.png"); plt.show()

## 13. Sensitivity analysis — Isolation Forest
Flag rate (1 / 3 / 5 %) × detection rule (3 consecutive / 3 of 5). Reported descriptively, not tested.

In [ ]:
rows = []
for c in CONTAMINATION_LEVELS:
    fc = f'flag_iso_{int(c*100):02d}'
    for rule, label in [('3consec', '3 consecutive'), ('3of5', '3 of 5')]:
        u = evaluate_detector(df_eval, df_eval[fc], df_eval[f'{fc}_{rule}'], f'{c:.0%} / {label}')
        rows.append({**summarise(u), 'flag_rate_pct': c * 100, 'rule': label,
                     'modems_detected': df_eval.loc[df_eval[f'{fc}_{rule}'], 'modem_key'].nunique()})
sensitivity = save_table(pd.DataFrame(rows).set_index('detector').round(1), 'T9_sensitivity_IF')
sensitivity[['units', 'detected_pct', 'early_within_pct', 'early_beyond_pct', 'concurrent_pct',
             'late_pct', 'not_detected_pct', 'median_lead_min', 'modems_detected']]

## 14. Missed degradation — Isolation Forest (primary configuration)
For units the model never detected: which threshold was breached and for how long, compared with detected units.

In [ ]:
br = df_eval.loc[df_eval['stig_any'], GROUP_KEYS + ['stig_utx', 'stig_usnr', 'stig_drx', 'stig_latency']]
per_unit = br.groupby(GROUP_KEYS).agg(breach_intervals=('stig_utx', 'size'),
                                      UsTX=('stig_utx', 'any'), UsSNR=('stig_usnr', 'any'),
                                      DsRX=('stig_drx', 'any'), Latency=('stig_latency', 'any'))
per_unit['breach_minutes'] = per_unit['breach_intervals'] * 5
per_unit['group'] = np.where(u_if.reindex(per_unit.index)['outcome'] == 'not_detected', 'missed', 'detected')

missed_table = per_unit.groupby('group').agg(
    units=('breach_minutes', 'size'),
    pct_UsTX=('UsTX', 'mean'), pct_UsSNR=('UsSNR', 'mean'),
    pct_DsRX=('DsRX', 'mean'), pct_Latency=('Latency', 'mean'),
    median_breach_minutes=('breach_minutes', 'median'))
for col in ['pct_UsTX', 'pct_UsSNR', 'pct_DsRX', 'pct_Latency']:
    missed_table[col] *= 100
missed_table = save_table(missed_table.round(1), 'T10_missed_degradation_IF')
print("Share of units involving each parameter (a unit can breach several) and median breach duration:")
missed_table

## 15. Retention log & save results
Requires `pip install pyarrow`. All tables (T1–T10) and figures (F1–F4) are already in `RESULTS_DIR`.

In [ ]:
retention_table = save_table(pd.DataFrame(retention_log), 'T0_retention_log')
print(retention_table.to_string(index=False))

keep_cols = (['modem_key', 'window', 'split', 'datetime'] + FEATURE_COLS + DEV_COLS + ['anomaly_score']
             + [c for c in df_model.columns if c.startswith(('flag_iso_', 'stig_'))])
df_model[keep_cols].to_parquet(BASE + "model_results.parquet", index=False)
pd.concat(results.values()).reset_index().to_parquet(BASE + "unit_outcomes_by_algorithm.parquet", index=False)
pd.concat([u_if_raw, u_if, u_if_3of5]).reset_index().to_parquet(BASE + "unit_outcomes_IF_rules.parquet", index=False)
print("Saved parquet files and", len(os.listdir(RESULTS_DIR)), "tables/figures in", RESULTS_DIR)

## 16. Exploratory analysis (post hoc) — sustained degradation and alert efficiency
**Not part of the confirmatory tests in Sections 9.2 / 9.6.** Defined after the main results showed that half of the
missed units were single five-minute breaches (median breach length 5 min, mostly UsSNR).

- **16.1 Sustained breaches:** the reference event is the start of the first run of **three consecutive breaching
  intervals** (15 min), i.e. the same persistence the detection rule requires. Units = test modem-windows with at least
  one such run. Lead time is measured to the start of that run.
- **16.2 Alert efficiency:** how many degradation events each algorithm anticipates per 100 modems it alerts the support team to.

In [ ]:
# ---- 16.1 Sustained breaches: reference = start of the first run of 3 consecutive breaching intervals
ev = df_eval[GROUP_KEYS + ['datetime', 'stig_any']].copy()
ev['stig_sustained'] = rule_consecutive(ev, 'stig_any', n=3)            # True on the 3rd breaching interval
ev['run_start'] = ev.groupby(GROUP_KEYS)['datetime'].shift(2)           # first interval of that run
sust_units = (ev.loc[ev['stig_sustained']].groupby(GROUP_KEYS)['run_start'].min()
              .rename('breach_time').to_frame())
print(f"Units with a sustained breach (>= 15 min): {len(sust_units):,} of {len(units):,} breaching units "
      f"({len(sust_units) / len(units) * 100:.1f} %)")

results_sust = {}
for name, fd in test_flags.items():
    d = ev[GROUP_KEYS + ['datetime']].join(fd)
    results_sust[name] = evaluate_detector(d, d['flag'], d['det'], name, unit_frame=sust_units)

sust_table = save_table(pd.DataFrame([summarise(u) for u in results_sust.values()])
                        .set_index('detector').round(1), 'T11_sustained_breaches_by_algorithm')
sust_table

In [ ]:
# Exploratory tests on sustained units (same tests as Section 11; reported as exploratory, not as hypothesis tests)
def tests_for(res):
    cnt = pd.DataFrame({n: u['outcome'].value_counts().reindex(OUTCOMES, fill_value=0) for n, u in res.items()}).T
    cu = cnt.loc[:, cnt.sum() > 0]
    chi2, p, dof, _ = chi2_contingency(cu)
    v = np.sqrt(chi2 / (cu.values.sum() * (min(cu.shape) - 1)))
    rows = [{'comparison': 'all three', 'test': 'Chi-square', 'statistic': chi2, 'p_value': p, 'effect': f"Cramer's V = {v:.3f}"}]
    a_all = res['IsolationForest']
    for other in ['KMeans', 'LOF']:
        b_all = res[other].reindex(a_all.index)
        a, b = a_all['outcome'] == 'early_within', b_all['outcome'] == 'early_within'
        tab = [[int((a & b).sum()), int((a & ~b).sum())], [int((~a & b).sum()), int((~a & ~b).sum())]]
        exact = (tab[0][1] + tab[1][0]) < 25
        r = mcnemar(tab, exact=exact, correction=not exact)
        rows.append({'comparison': f'IF vs {other}', 'test': 'McNemar (early within bound)', 'statistic': r.statistic,
                     'p_value': r.pvalue, 'effect': f'IF only {tab[0][1]} / {other} only {tab[1][0]}'})
        both = a_all['det_time'].notna() & b_all['det_time'].notna()
        dlt = a_all.loc[both, 'lead_min'] - b_all.loc[both, 'lead_min']
        nz = dlt[dlt != 0]
        if len(nz):
            w, pw = wilcoxon(a_all.loc[both, 'lead_min'], b_all.loc[both, 'lead_min'], zero_method='wilcox')
            rk = nz.abs().rank(); rrb = (rk[nz > 0].sum() - rk[nz < 0].sum()) / rk.sum()
        else:
            w, pw, rrb = np.nan, 1.0, 0.0
        rows.append({'comparison': f'IF vs {other}', 'test': 'Wilcoxon (lead time)', 'statistic': w, 'p_value': pw,
                     'effect': f'n = {int(both.sum())}, median diff = {dlt.median():.1f} min, r_rb = {rrb:.3f}'})
    return cnt, pd.DataFrame(rows)

sust_counts, sust_tests = tests_for(results_sust)
save_table(sust_counts, 'T12_sustained_outcome_counts')
save_table(sust_tests, 'T13_sustained_exploratory_tests')
display(sust_counts); sust_tests

In [ ]:
# IF early detections by lead time on sustained units (same buckets as Section 9.1)
sust_bucket_table = save_table(bucket_table_for(results_sust['IsolationForest']), 'T14_sustained_buckets_IF')
sust_bucket_table

In [ ]:
# ---- 16.2 Alert efficiency: anticipated degradation per 100 alerted modems
rows = []
for name, fd in test_flags.items():
    d = ev[['modem_key']].join(fd)
    alerted = d.loc[d['det'], 'modem_key'].nunique()
    for label, res in [('all breaches', results), ('sustained breaches', results_sust)]:
        u = res[name]
        early = (u['outcome'] == 'early_within').sum()
        credible = ((u['outcome'] == 'early_within') & (u['persistence'] >= 0.5)).sum()
        rows.append({'algorithm': name, 'reference': label, 'modems_alerted': alerted,
                     'early_within': int(early), 'credible_early (persistence >= 0.5)': int(credible),
                     'early_per_100_alerted': early / alerted * 100,
                     'credible_per_100_alerted': credible / alerted * 100})
efficiency = save_table(pd.DataFrame(rows).set_index(['reference', 'algorithm']).round(2), 'T15_alert_efficiency')
efficiency

In [ ]:
# ---- F5: outcome shares on sustained units
share_s = sust_counts.div(sust_counts.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(len(share_s))
for o in OUTCOMES:
    vals = share_s[o].to_numpy()
    ax.barh(share_s.index, vals, left=left, color=COL_OUTCOME[o], edgecolor='white', linewidth=2,
            height=0.6, label=OUTCOME_LABELS[o])
    for y, (l, v) in enumerate(zip(left, vals)):
        if v >= 5:
            ax.text(l + v / 2, y, f'{v:.0f}%', ha='center', va='center', fontsize=9,
                    color='white' if o in ('early_within', 'early_beyond') else INK)
    left += vals
ax.set_xlim(0, 100); ax.set_xlabel('Share of units (%)'); ax.invert_yaxis(); ax.grid(axis='y', visible=False)
ax.set_title('Exploratory: detection outcome for sustained breaches (\u2265 15 min)', loc='left', color=INK, fontsize=10)
ax.legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.22), frameon=False, fontsize=9)
plt.savefig(RESULTS_DIR + "F5_outcome_shares_sustained.png"); plt.show()

In [ ]:
# ---- 16.3 Splitting "late": rule latency vs. genuinely late (descriptive breakdown of the existing category)
# The 3-consecutive rule fires on the 3rd flagged interval, so a model that starts flagging when the breach starts
# reports L = -10 min. Detections up to 15 min after the breach are therefore treated as detection at onset.
ONSET_WINDOW_MIN = 15

def late_split(u):
    det = u['det_time'].notna()
    L = u['lead_min']
    return {
        'units': len(u),
        'early_within_pct':   (u['outcome'] == 'early_within').mean() * 100,
        'early_beyond_pct':   (u['outcome'] == 'early_beyond').mean() * 100,
        'concurrent_pct':     (u['outcome'] == 'concurrent').mean() * 100,
        'late_onset_pct':     (det & (L < 0) & (L >= -ONSET_WINDOW_MIN)).mean() * 100,   # within 15 min after breach
        'late_later_pct':     (det & (L < -ONSET_WINDOW_MIN)).mean() * 100,              # more than 15 min after
        'not_detected_pct':   (~det).mean() * 100,
        'at_or_near_onset_pct': (det & (L <= 0) & (L >= -ONSET_WINDOW_MIN)).mean() * 100, # concurrent + late_onset
        'median_late_later_min': L[det & (L < -ONSET_WINDOW_MIN)].median(),
    }

rows = []
for label, res in [('all breaches', results), ('sustained breaches', results_sust)]:
    for name, u in res.items():
        rows.append({'reference': label, 'algorithm': name, **late_split(u)})
late_split_table = save_table(pd.DataFrame(rows).set_index(['reference', 'algorithm']).round(1), 'T16_late_split')
late_split_table

---
# Appendix — diagnostics behind the design decisions
Not required for a normal run. Some cells need intermediate frames — run them where noted.

### A1. Raw null concentration (justifies row-level dropping in §2)
*Run between §1 and §2 (needs `df`).*

In [ ]:
for col in ['utxmax', 'usnrmin', 'drxmin', 'drxmax']:
    per_modem = df[df[col].isnull()].groupby('modem_key').size()
    print(f"{col:<8} nulls={df[col].isnull().sum():>8,}  modems affected={len(per_modem):>6,}  "
          f"median nulls/modem={per_modem.median():.0f}")

### A2. Per-modem variance (justifies signed absolute deviation for UsTX in §4)

In [ ]:
nuniq = df_model.groupby(GROUP_KEYS)[FEATURE_COLS].nunique()
print("Share of modem-windows with a single constant value:")
print((nuniq == 1).mean().mul(100).round(1).astype(str) + " %")

### A3. Feature drivers — which features separate flagged from unflagged rows (test window)

In [ ]:
flagged = df_eval[PRIMARY_FLAG]
drivers = pd.DataFrame({'flagged_mean':   df_eval.loc[flagged,  MODEL_FEATURES].mean(),
                        'unflagged_mean': df_eval.loc[~flagged, MODEL_FEATURES].mean()})
drivers['abs_diff'] = (drivers['flagged_mean'] - drivers['unflagged_mean']).abs()
save_table(drivers.sort_values('abs_diff', ascending=False).round(2), 'TA3_feature_drivers')

### A4. Flag concentration per modem (test window)

In [ ]:
per_modem_flags = df_eval[df_eval[PRIMARY_FLAG]].groupby('modem_key').size()
print(f"Modems with >=1 flag: {len(per_modem_flags):,} of {df_eval['modem_key'].nunique():,}")
print(per_modem_flags.describe().round(1))

### A5. Latency population split (justifies excluding absolute latency)

In [ ]:
low = df_model['latencyavg'] < 1000
share_low = (df_model[low].groupby('modem_key').size()
             / df_model.groupby('modem_key').size()).dropna()
low_modems = share_low[share_low > 0.9].index.tolist()
print(f"Modems >90% low-latency: {len(low_modems):,}")

# SQL to check model/firmware of these modems in MariaDB
print(f"SELECT model, firmware, COUNT(*) n FROM modemids2 WHERE id IN "
      f"({','.join(map(str, low_modems))}) GROUP BY model, firmware ORDER BY n DESC;")

### A6. Containment check — are detected modem-windows a subset of Stig-breaching ones?

In [ ]:
det_mw  = set(first_time(df_eval, df_eval[f'{PRIMARY_FLAG}_3consec']).index)
raw_mw  = set(first_time(df_eval, df_eval[PRIMARY_FLAG]).index)
stig_mw = set(units.index)
print(f"Detected (3-consec) but never breach a threshold: {len(det_mw - stig_mw):,}")
print(f"Breach a threshold but never detected (3-consec): {len(stig_mw - det_mw):,}   <- false negatives")
print(f"Raw-flagged but never breach a threshold:         {len(raw_mw - stig_mw):,}")